# Threshold classification

## Import libraries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Confusion Matrix

In [2]:
def matriz_conf(y_true, y_pred, labels):
    total_labels = labels
    print(total_labels)
    cm = np.zeros((len(total_labels),len(total_labels)), dtype=int)
    for i in range(len(y_true)):
        cm[y_true[i]][y_pred[i]] += 1

    
    cm = pd.DataFrame(cm, columns=total_labels, index=total_labels)

    cm_transp = pd.DataFrame(np.transpose(cm.to_numpy()), columns=total_labels, index=total_labels)

    for c in cm_transp.columns:
        cm_transp[c] = cm_transp[c]/cm_transp[c].sum()

    cm_porcento = pd.DataFrame(np.transpose(cm_transp.to_numpy()), columns=total_labels, index=total_labels)

    return cm, cm_porcento

def acc(cm, hidden_classes):
    cm_transp = pd.DataFrame(np.transpose(cm.dropna().to_numpy()), columns=cm.columns, index=cm.columns)
    acc = 0
    total = 0
    for c in cm_transp.columns:
        if c not in hidden_classes:
            acc += cm_transp[c][c]
        else:
            acc += cm_transp[c][-1]
        total += cm_transp[c].sum()
    return acc/total

In [ ]:
filenames = [0,2,3,4,5]

labels_str = ['DDoS', 'Benign', 'DoS', 'BruteForce', 'Bot', 'Web']

filenames

pd.set_option('future.no_silent_downcasting', True)

[0, 2, 3, 4, 5]

In [ ]:
exp_test = []
y_true_all_exp_test = []
for i in range(len(filenames)):
    exp_test.append(pd.read_csv(f'test_{filenames[i]}_GMM_BIC_1_10_scores.csv'))
    y_true_all_exp_test.append(exp_test[i]['Label'].values.tolist())
    exp_test[i] = exp_test[i].drop(columns=['Label'])

In [ ]:
from sklearn.metrics import classification_report
ths = [21]
f1s = [[]]
for i in range(len(exp_test)):
    index = 0
    for th in ths:
        y_pred = []
        prog= 0
        for j in range(len(exp_test[i])):
            m = np.nanmax(exp_test[i].values[j])
            if m > th:
                y_pred.append(exp_test[i].values[j].tolist().index(m))
            else:
                y_pred.append(-1)

        cm, cm_porcento = matriz_conf(y_true_all_exp_test[i], y_pred, list(range(len(labels_str))) + [-1])
        print(f'th = {th} hidden = {filenames[i]}')
        display(cm)
        tp = cm[-1][filenames[i]]
        fp = cm[-1].sum() - tp
        fn = cm.iloc[filenames[i]].sum() - tp
        tn = cm.drop(columns=[-1]).values.sum() - fn

        acc = (tp+tn)/(tp+fp+tn+fn)
        recall = tp/(tp+fn)
        precision = tp/(tp+fp)
        if precision == 0 or recall == 0:
            f1 = 0
        else:
            f1 = 2*precision*recall/(precision+recall)

        f1s[index].append(f1)
        index += 1
        print(f'th: {th} hidden: {filenames[i]} acc:{acc} recall:{recall} precision:{precision} f1:{f1}')

        true_labels = np.array(y_true_all_exp_test[i])
        true_labels[true_labels == filenames[i]] = -1

        print('MULTICLASS DETECTION')
        print(classification_report(true_labels, y_pred))